In [23]:
#Importing
import pandas as pd
import numpy as np
from datascience import *

#Reading files
FCT = pd.read_csv('Tanzania - Hicks - FCT.csv')
Household_characteristics = pd.read_csv('Tanzania - Household Characteristics.csv')

#Reading files for 2019 - 2020
Food_Expenditure_1920 = pd.read_csv('Tanzania - Food Expenditures (2019-20).csv')
Food_Prices_1920 = pd.read_csv('Tanzania - Food Prices (2019-20).csv').dropna(how='any')
Household_characteristics_1920 = Household_characteristics[Household_characteristics['t'] == "2019-20"]

#Reading files for 2020 - 2021
Food_Expenditure_2021 = pd.read_csv('Tanzania - Food Expenditures (2020-21).csv')
Food_Prices_2021 = pd.read_csv('Tanzania - Food Prices (2020-21).csv').dropna(how='any')
Household_characteristics_2021 = Household_characteristics[Household_characteristics['t'] == "2020-21"]


#Total nutrient for each household
def ttl_nutrient_household(FCT, Household_Characteristic, Expenditure, Food_Price):
    
    #Merging datasets
    merged_df = pd.merge(Household_Characteristic, Expenditure.drop(columns=['t', 'm']), on='i', how='inner')
    merged_df = pd.merge(merged_df, Food_Price, on=["j","m"],how = "left")
    merged_df["Quantity of Food"] = merged_df["Expenditure"] / merged_df["Price"]
    merged_df = merged_df.drop_duplicates(subset=['i', 'j'], keep='first')
    merged_df = merged_df[["i", "m","log HSize", "j","u","Quantity of Food"]]
    merged_df = pd.merge(merged_df, FCT, on='j', how='left')
    
    df_total_nutrients_per_household = merged_df[[
        "i",
        "Quantity of Food",
        "Energy",
        "Protein",
        "Vitamin A",
        "Vitamin D",
        "Vitamin E",
        "Vitamin C",
        "Vitamin B-6",
        "Vitamin B-12",
        "Calcium",
        "Magnesium",
        "Iron",
        "Zinc",
        "Fiber"
    ]]
    
    nutrient_cols = [
        "Energy", "Protein", "Vitamin A", "Vitamin D", "Vitamin E",
        "Vitamin C", "Vitamin B-6", "Vitamin B-12",
        "Calcium", "Magnesium", "Iron", "Zinc", "Fiber"
    ]
    
    #Multiplying nutrients by quantity
    df_total_nutrients_per_household [nutrient_cols] = df_total_nutrients_per_household [nutrient_cols].multiply(df_total_nutrients_per_household ["Quantity of Food"], axis=0)
    
    #Grouping individuals
    df_total_nutrients_per_household = df_total_nutrients_per_household.groupby("i")[nutrient_cols].sum().reset_index()[["i"]+nutrient_cols]
    
    return df_total_nutrients_per_household




#Average nutrient for each person in the household
def avg_nutrient_household(FCT, Household_Characteristic, Expenditure, Food_Price):
    ttl = ttl_nutrient_household(FCT, Household_Characteristic, Expenditure, Food_Price)
    ttl_w_household = pd.merge(ttl, Household_Characteristic[["i","log HSize"]], on=["i"], how="left")

    #Create HSize from log HSize
    ttl_w_household["HSize"] = np.exp(
        ttl_w_household["log HSize"]
    )

    #Drop the old column
    ttl_w_household = ttl_w_household.drop(
        columns=["log HSize"]
    )

    nutrient_cols = [
    "Energy", "Protein", "Vitamin A", "Vitamin D", "Vitamin E",
    "Vitamin C", "Vitamin B-6", "Vitamin B-12",
    "Calcium", "Magnesium", "Iron", "Zinc", "Fiber"
    ]

    # divide each nutrient by household size
    ttl_w_household.loc[:, nutrient_cols] = (
        ttl_w_household[nutrient_cols]
        .div(ttl_w_household["HSize"], axis=0)
    )

    return ttl_w_household

avg_nutrient_household(FCT, Household_characteristics_1920, Food_Expenditure_1920, Food_Prices_1920)

/tmp/ipykernel_70/3321388570.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_total_nutrients_per_household [nutrient_cols] = df_total_nutrients_per_household [nutrient_cols].multiply(df_total_nutrients_per_household ["Quantity of Food"], axis=0)


,i,Energy,Protein,Vitamin A,Vitamin D,Vitamin E,Vitamin C,Vitamin B-6,Vitamin B-12,Calcium,Magnesium,Iron,Zinc,Fiber,HSize
0,0001-001-001,15552.109109,441.845463,11342.893335,0.000000,16.759279,353.935465,11.738670,1.374486,11195.229724,2381.858800,147.869566,62.554246,502.660158,2.0
1,0001-001-003,27278.593792,737.883151,20162.542476,0.000000,39.143130,634.126183,22.529017,13.798042,14604.888896,5210.375760,283.229777,130.939004,714.259104,1.0
2,0001-001-004,32742.276337,934.186206,23456.156702,100.974998,77.217637,1409.537846,29.420702,32.077825,3677.333615,7138.730485,234.479985,149.447751,350.793132,1.0
3,0001-004-001,17549.739139,398.512031,14399.051335,0.000000,48.952738,220.738267,9.534897,7.237517,7014.933426,2350.531687,99.662609,50.584028,298.946311,3.0
4,0001-004-002,15288.622904,289.108197,5849.700626,0.000000,9.759384,83.204892,9.474190,0.000000,11541.636177,1656.659741,128.849651,51.838746,535.094517,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1105,0856-001-001,3767.097792,54.791194,1673.638455,17.289531,2.360593,43.705586,0.673890,1.584874,758.157120,234.780452,6.727309,4.669047,84.762977,11.0
1106,0856-001-003,6826.623824,159.269359,99.608380,27.169264,5.355353,47.623505,2.863006,2.490516,433.598129,778.435862,16.531044,15.268848,58.340132,7.0
1107,0857-001-001,7845.502221,197.528578,553.639698,35.659659,14.344220,113.503022,4.462810,4.470939,1685.295138,1014.270814,29.829937,20.527748,135.224364,8.0
1108,0858-001-001,12559.379633,345.306024,11066.851518,21.036458,35.717609,150.298171,10.517559,1.928342,2278.760466,4387.491926,141.932181,64.979109,227.324742,3.0


In [16]:
ttl_nutrient_household(FCT, Household_characteristics_1920, Food_Expenditure_1920, Food_Prices_1920)

/tmp/ipykernel_70/449772168.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_total_nutrients_per_household [nutrient_cols] = df_total_nutrients_per_household [nutrient_cols].multiply(df_total_nutrients_per_household ["Quantity of Food"], axis=0)


,i,Energy,Protein,Vitamin A,Vitamin D,Vitamin E,Vitamin C,Vitamin B-6,Vitamin B-12,Calcium,Magnesium,Iron,Zinc,Fiber
0,0001-001-001,31104.218220,883.690926,22685.786670,0.000000,33.518559,707.870931,23.477341,2.748972,22390.459449,4763.717599,295.739132,125.108493,1005.320315
1,0001-001-003,27278.593792,737.883151,20162.542476,0.000000,39.143130,634.126183,22.529017,13.798042,14604.888896,5210.375760,283.229777,130.939004,714.259104
2,0001-001-004,32742.276337,934.186206,23456.156702,100.974998,77.217637,1409.537846,29.420702,32.077825,3677.333615,7138.730485,234.479985,149.447751,350.793132
3,0001-004-001,52649.217434,1195.536094,43197.154018,0.000000,146.858214,662.214801,28.604690,21.712552,21044.800285,7051.595062,298.987828,151.752084,896.838933
4,0001-004-002,15288.622904,289.108197,5849.700626,0.000000,9.759384,83.204892,9.474190,0.000000,11541.636177,1656.659741,128.849651,51.838746,535.094517
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1105,0856-001-001,41438.075722,602.703137,18410.023005,190.184846,25.966519,480.761450,7.412793,17.433611,8339.728320,2582.584972,74.000399,51.359518,932.392746
1106,0856-001-003,47786.366768,1114.885510,697.258659,190.184846,37.487471,333.364533,20.041041,17.433611,3035.186904,5449.051037,115.717308,106.881934,408.380924
1107,0857-001-001,62764.017792,1580.228627,4429.117582,285.277268,114.753760,908.024180,35.702477,35.767513,13482.361111,8114.166517,238.639493,164.221986,1081.794915
1108,0858-001-001,37678.138911,1035.918071,33200.554565,63.109374,107.152826,450.894513,31.552677,5.785026,6836.281400,13162.475784,425.796542,194.937328,681.974225


In [17]:
ttl_nutrient_household(FCT, Household_characteristics_2021, Food_Expenditure_2021, Food_Prices_2021)

/tmp/ipykernel_70/449772168.py:56: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_total_nutrients_per_household [nutrient_cols] = df_total_nutrients_per_household [nutrient_cols].multiply(df_total_nutrients_per_household ["Quantity of Food"], axis=0)


,i,Energy,Protein,Vitamin A,Vitamin D,Vitamin E,Vitamin C,Vitamin B-6,Vitamin B-12,Calcium,Magnesium,Iron,Zinc,Fiber
0,1009-001-01,60316.113238,1656.464898,44774.303874,24.000000,74.978714,680.502863,33.285703,91.186949,17078.523862,8279.226294,442.152154,216.951469,491.746926
1,1025-001-02,37644.147560,853.302252,14671.377306,0.000000,68.784202,488.094303,24.909772,5.410169,3971.731639,7323.112189,277.870237,144.724232,520.056023
2,1039-001-01,66858.369765,1586.417496,10371.751671,58.148112,60.871251,216.896005,29.976597,5.330244,4361.085118,12050.898639,425.510168,264.196876,723.639308
3,1078-001-01,60054.528225,1331.538155,10090.158533,56.843526,379.336528,3352.186847,66.680111,33.787485,12636.907167,11371.375189,331.108889,181.632981,940.245384
4,1102-001-01,29695.664089,469.083635,6073.468307,0.000000,50.485167,64.387266,13.151144,11.430731,4563.869740,5865.774792,157.881350,81.332668,291.654656
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
401,9760-001-99,50445.369370,946.586846,12783.513526,0.000000,57.970216,1928.008653,45.446446,0.000000,4895.839825,10732.184286,438.348383,225.067902,611.396210
402,9772-001-99,36409.058428,801.840447,2939.809259,93.036979,64.490291,634.746866,24.481975,8.528390,1806.754368,8397.697402,218.764956,130.249545,442.161503
403,9784-001-99,27429.454809,474.375492,15484.289307,20.933320,67.754580,1733.462409,22.711818,1.918888,4280.254955,7471.438313,234.284401,106.503028,438.699052
404,9796-001-99,44613.312859,1376.330435,15390.700556,279.110936,120.288528,175.708828,36.786568,25.585169,4285.082309,14470.166080,374.064382,196.380122,642.725141
